# Análise de Eventos de Chutes - RoboCup (Dados Unificados)

Este notebook processa os dados unificados de chutes da RoboCup e gera dois tipos de CSVs:
1. **CSV de Features Consolidadas**: Uma linha por chute com informações da bola, chutador, goleiro e features calculadas
2. **CSV de Posições dos Jogadores**: Múltiplas linhas por chute com posições de todos os robôs no campo

**Nota**: Todos os eventos de todos os jogos estão unificados em uma única pasta. Arquivos de referee, commands e imagens/vídeos são ignorados.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurações - Dados Unificados
DADOS_DIR = Path('Dados finais')
CHUTES_FILE = DADOS_DIR / 'Chutes' / 'all_events.csv'
EVENTOS_DIR = DADOS_DIR / 'Eventos'
OUTPUT_DIR = Path('Dados_Processados_Finais')
OUTPUT_DIR.mkdir(exist_ok=True)

# Dimensões do campo (em mm) - configurações oficiais
# Campo: X de -6000 a +6000 (12000mm total), Y de -4500 a +4500 (9000mm total)
FIELD_X_MIN = -6000
FIELD_X_MAX = 6000
FIELD_Y_MIN = -4500
FIELD_Y_MAX = 4500
FIELD_LENGTH = FIELD_X_MAX - FIELD_X_MIN  # 12000 mm (12 m)
FIELD_WIDTH = FIELD_Y_MAX - FIELD_Y_MIN    # 9000 mm (9 m)
GOAL_WIDTH = 1800     # Largura do gol (1.8 m)
GOAL_DEPTH = 180      # Profundidade do gol (180 mm)
GOAL_Y = 0            # Centro do gol em Y (campo centrado em 0)

print("✓ Bibliotecas importadas")
print(f"✓ Dados Finais - Configuração:")
print(f"  - Chutes: {CHUTES_FILE}")
print(f"  - Eventos: {EVENTOS_DIR}")
print(f"  - Saída: {OUTPUT_DIR}")
print(f"\n⚠ Nota: Arquivos de referee, commands e imagens/vídeos serão ignorados")

✓ Bibliotecas importadas
✓ Dados Finais - Configuração:
  - Chutes: Dados finais\Chutes\all_events.csv
  - Eventos: Dados finais\Eventos
  - Saída: Dados_Processados_Finais

⚠ Nota: Arquivos de referee, commands e imagens/vídeos serão ignorados


## 2. Funções para Cálculo de Features Geométricas

In [2]:
def calcular_features_geometricas(ball_x, ball_y, robot_x, robot_y, robot_w, 
                                   goalkeeper_x, goalkeeper_y, all_opponents_positions,
                                   cor_jogador_chutou):
    """
    Calcula as features geométricas para um evento de chute.
    
    Parâmetros:
    - ball_x, ball_y: Posição da bola
    - robot_x, robot_y, robot_w: Posição e orientação do chutador
    - goalkeeper_x, goalkeeper_y: Posição do goleiro
    - all_opponents_positions: Lista de tuplas (x, y) dos oponentes
    - cor_jogador_chutou: 'blue' ou 'yellow' (para determinar qual gol é o alvo)
    
    Retorna: Dicionário com as features calculadas
    """
    features = {}
    
    # Determinar qual gol é o alvo baseado na posição do GOLEIRO
    # O goleiro defende seu próprio gol, então o time chutador ataca o gol ONDE está o goleiro
    # Se goalkeeper_x > 0 (goleiro na direita), o chutador ataca o gol da direita (+X)
    # Se goalkeeper_x < 0 (goleiro na esquerda), o chutador ataca o gol da esquerda (-X)
    if not np.isnan(goalkeeper_x):
        # Usar posição do goleiro para determinar qual gol atacar
        if goalkeeper_x > 0:
            # Goleiro na direita (+X), chutador ataca o gol da direita (+X)
            goal_x = FIELD_LENGTH / 2  # +6000
        else:
            # Goleiro na esquerda (-X), chutador ataca o gol da esquerda (-X)
            goal_x = -FIELD_LENGTH / 2  # -6000
    else:
        # Fallback: se não há goleiro identificado, usar lógica simples
        # Assumir que a bola está mais perto do gol que vai atacar
        if ball_x > 0:
            goal_x = FIELD_LENGTH / 2  # Atacando direita
        else:
            goal_x = -FIELD_LENGTH / 2  # Atacando esquerda
    
    goal_y = GOAL_Y  # Centro do campo em Y
    
    # 1. feat_dist_goal: Distância da bola ao centro do gol
    features['feat_dist_goal'] = np.sqrt((ball_x - goal_x)**2 + (ball_y - goal_y)**2)
    
    # 2. feat_angle_goal: Ângulo da bola para o centro do gol
    angle_to_goal_rad = np.arctan2(goal_y - ball_y, goal_x - ball_x)
    features['feat_angle_goal'] = np.degrees(angle_to_goal_rad)
    
    # 3. feat_visible_angle: Ângulo de visão (abertura) para o gol
    goal_post_left_y = goal_y - GOAL_WIDTH / 2
    goal_post_right_y = goal_y + GOAL_WIDTH / 2
    
    angle_to_left_post = np.arctan2(goal_post_left_y - ball_y, goal_x - ball_x)
    angle_to_right_post = np.arctan2(goal_post_right_y - ball_y, goal_x - ball_x)
    visible_angle_rad = abs(angle_to_right_post - angle_to_left_post)
    features['feat_visible_angle'] = np.degrees(visible_angle_rad)
    
    # 4. feat_min_dist_opponent: Distância para o marcador mais próximo
    if len(all_opponents_positions) > 0:
        distances = [np.sqrt((ball_x - opp_x)**2 + (ball_y - opp_y)**2) 
                     for opp_x, opp_y in all_opponents_positions]
        features['feat_min_dist_opponent'] = min(distances)
    else:
        features['feat_min_dist_opponent'] = np.nan
    
    # 5. feat_opponents_in_cone: Quantidade de oponentes bloqueando o gol
    opponents_in_cone = 0
    cone_half_angle = visible_angle_rad / 2 + np.radians(10)  # Margem de 10 graus
    
    for opp_x, opp_y in all_opponents_positions:
        dist_ball_to_opp = np.sqrt((opp_x - ball_x)**2 + (opp_y - ball_y)**2)
        
        if dist_ball_to_opp < features['feat_dist_goal']:
            angle_to_opponent = np.arctan2(opp_y - ball_y, opp_x - ball_x)
            angle_diff = abs(angle_to_opponent - angle_to_goal_rad)
            
            if angle_diff > np.pi:
                angle_diff = 2 * np.pi - angle_diff
            
            if angle_diff <= cone_half_angle:
                opponents_in_cone += 1
    
    features['feat_opponents_in_cone'] = opponents_in_cone
    
    # 6. feat_robot_orientation: Para onde o robô estava olhando (em graus)
    features['feat_robot_orientation'] = np.degrees(robot_w)
    
    # Feature adicional: Diferença entre orientação do robô e direção ao gol
    angle_diff_orientation = angle_to_goal_rad - robot_w
    while angle_diff_orientation > np.pi:
        angle_diff_orientation -= 2 * np.pi
    while angle_diff_orientation < -np.pi:
        angle_diff_orientation += 2 * np.pi
    features['feat_orientation_to_goal_diff'] = np.degrees(angle_diff_orientation)
    
    return features

print("✓ Função calcular_features_geometricas criada (usa posição do goleiro para determinar gol)")

✓ Função calcular_features_geometricas criada (usa posição do goleiro para determinar gol)


## 3. Função para Processar CSV #1 - Features Consolidadas

In [3]:
def processar_csv_features_consolidadas(kicks_csv_path, eventos_folder):
    """
    Processa eventos unificados e cria o CSV #1 com features consolidadas.
    Uma linha por evento de chute.
    
    Arquivos ignorados: referee.csv, imagens e vídeos
    """
    print(f"\n{'='*60}")
    print(f"Processando Dados Unificados")
    print(f"{'='*60}")
    
    # Carregar eventos de chutes
    kicks_df = pd.read_csv(kicks_csv_path)
    print(f"✓ {len(kicks_df)} eventos de chute carregados")
    
    consolidated_data = []
    
    for idx, kick_row in kicks_df.iterrows():
        kick_event = kick_row['kick_event']
        timestamp_ball = kick_row['timestamp_ball']
        
        try:
            # 1. Carregar dados da bola
            ball_file = eventos_folder / f"{kick_event}_ball.csv"
            if not ball_file.exists():
                print(f"  ⚠ Arquivo não encontrado: {ball_file.name}")
                continue
                
            ball_df = pd.read_csv(ball_file)
            ball_df['timestamp_diff'] = abs(ball_df['timestamp'] - timestamp_ball)
            closest_ball_idx = ball_df['timestamp_diff'].idxmin()
            ball_data = ball_df.loc[closest_ball_idx]
            
            # 2. Carregar dados dos robôs processados
            robots_file = eventos_folder / f"{kick_event}_processed_robots.csv"
            if not robots_file.exists():
                print(f"  ⚠ Arquivo não encontrado: {robots_file.name}")
                continue
                
            robots_df = pd.read_csv(robots_file)
            robots_df['timestamp_diff'] = abs(robots_df['timestamp'] - ball_data['timestamp'])
            
            # Encontrar o chutador - descobrir automaticamente em qual team está
            # IMPORTANTE: processed_robots usa 'robots_blue' e 'robots_yellow', não 'allies'/'enemies'
            kicker_id = kick_row['id_jogador_chutou']
            kicker_color = kick_row['cor_jogador_chutou']
            kicker_team = f'robots_{kicker_color}'
            
            # Buscar dados do chutador
            kicker_data = robots_df[
                (robots_df['team'] == kicker_team) & 
                (robots_df['robot_id'] == kicker_id) &
                (robots_df['timestamp'] == ball_data['timestamp'])
            ].copy()
            
            if kicker_data.empty:
                kicker_data = robots_df[
                    (robots_df['team'] == kicker_team) & 
                    (robots_df['robot_id'] == kicker_id)
                ].copy()
                
                if kicker_data.empty:
                    print(f"  ⚠ Chutador não encontrado: {kick_event}, id={kicker_id}, cor={kicker_color}")
                    continue
                    
                kicker_data = kicker_data.loc[kicker_data['timestamp_diff'].idxmin()]
            else:
                kicker_data = kicker_data.iloc[0]
            
            # Encontrar o goleiro
            goalkeeper_id = kick_row['id_goleiro']
            goalkeeper_color = kick_row['cor_goleiro']
            goalkeeper_team = f'robots_{goalkeeper_color}'
            goalkeeper_data = None
            
            goalkeeper_data = robots_df[
                (robots_df['team'] == goalkeeper_team) & 
                (robots_df['robot_id'] == goalkeeper_id) &
                (robots_df['timestamp'] == ball_data['timestamp'])
            ].copy()
            
            if goalkeeper_data.empty:
                goalkeeper_data = robots_df[
                    (robots_df['team'] == goalkeeper_team) & 
                    (robots_df['robot_id'] == goalkeeper_id)
                ].copy()
                
                if not goalkeeper_data.empty:
                    goalkeeper_data = goalkeeper_data.loc[goalkeeper_data['timestamp_diff'].idxmin()]
                else:
                    goalkeeper_data = None
            else:
                goalkeeper_data = goalkeeper_data.iloc[0]
            
            # 3. Obter robôs em janela de tempo para features E contagem
            # IMPORTANTE: Usar mesma janela para ambos!
            time_window = 0.1
            robots_in_window = robots_df[robots_df['timestamp_diff'] <= time_window].copy()
            
            # Para cada robô, pegar apenas a última aparição (timestamp mais próximo)
            robots_at_time = robots_in_window.sort_values('timestamp_diff').groupby(['team', 'robot_id']).first().reset_index()
            
            # Time oponente ao chutador
            opponents_team = f'robots_{goalkeeper_color}'
            opponents_at_time = robots_at_time[robots_at_time['team'] == opponents_team]
            
            all_opponents_positions = list(zip(
                opponents_at_time['position_x'], 
                opponents_at_time['position_y']
            ))
            
            # 4. Calcular features geométricas
            features = calcular_features_geometricas(
                ball_x=ball_data['position_x'],
                ball_y=ball_data['position_y'],
                robot_x=kicker_data['position_x'],
                robot_y=kicker_data['position_y'],
                robot_w=kicker_data['position_w'],
                goalkeeper_x=goalkeeper_data['position_x'] if goalkeeper_data is not None else np.nan,
                goalkeeper_y=goalkeeper_data['position_y'] if goalkeeper_data is not None else np.nan,
                all_opponents_positions=all_opponents_positions,
                cor_jogador_chutou=kick_row['cor_jogador_chutou']
            )
            
            # 5. Contar robôs em campo (usar mesma janela e mesmos robôs)
            num_blue = len(robots_at_time[robots_at_time['team'] == 'robots_blue'])
            num_yellow = len(robots_at_time[robots_at_time['team'] == 'robots_yellow'])
            
            # 6. Construir linha de dados consolidada
            row_data = {
                'kick_event': kick_event,
                'timestamp': ball_data['timestamp'],
                'timestamp_original': timestamp_ball,
                'id_jogador_chutou': kicker_id,
                'cor_jogador_chutou': kick_row['cor_jogador_chutou'],
                'id_goleiro': goalkeeper_id,
                'cor_goleiro': kick_row['cor_goleiro'],
                'gol': kick_row['gol'],
                'ball_position_x': ball_data['position_x'],
                'ball_position_y': ball_data['position_y'],
                'ball_velocity_x': ball_data['velocity_x'],
                'ball_velocity_y': ball_data['velocity_y'],
                'ball_velocity_norm': ball_data['velocity_norm'],
                'ball_acceleration_x': ball_data['acceleration_x'],
                'ball_acceleration_y': ball_data['acceleration_y'],
                'ball_acceleration_norm': ball_data['acceleration_norm'],
                'kicker_position_x': kicker_data['position_x'],
                'kicker_position_y': kicker_data['position_y'],
                'kicker_position_w': kicker_data['position_w'],
                'kicker_velocity_x': kicker_data['velocity_x'],
                'kicker_velocity_y': kicker_data['velocity_y'],
                'kicker_velocity_norm': kicker_data['velocity_norm'],
                'goalkeeper_position_x': goalkeeper_data['position_x'] if goalkeeper_data is not None else np.nan,
                'goalkeeper_position_y': goalkeeper_data['position_y'] if goalkeeper_data is not None else np.nan,
                'goalkeeper_position_w': goalkeeper_data['position_w'] if goalkeeper_data is not None else np.nan,
                'goalkeeper_velocity_x': goalkeeper_data['velocity_x'] if goalkeeper_data is not None else np.nan,
                'goalkeeper_velocity_y': goalkeeper_data['velocity_y'] if goalkeeper_data is not None else np.nan,
                'goalkeeper_velocity_norm': goalkeeper_data['velocity_norm'] if goalkeeper_data is not None else np.nan,
                'num_blue_on_field': num_blue,
                'num_yellow_on_field': num_yellow,
            }
            
            row_data.update(features)
            consolidated_data.append(row_data)
            
        except Exception as e:
            print(f"  ✗ Erro ao processar {kick_event}: {str(e)}")
            continue
    
    result_df = pd.DataFrame(consolidated_data)
    print(f"\n✓ {len(result_df)} eventos processados com sucesso")
    print(f"✓ {len(result_df.columns)} colunas no dataset")
    
    return result_df

print("✓ Função processar_csv_features_consolidadas criada")

✓ Função processar_csv_features_consolidadas criada


## 4. Função para Processar CSV #2 - Posições dos Jogadores

In [4]:
def processar_csv_posicoes_jogadores(kicks_csv_path, eventos_folder):
    """
    Processa eventos unificados e cria o CSV #2 com posições de todos os jogadores.
    Múltiplas linhas por evento de chute (uma linha por robô).
    """
    print(f"\n{'='*60}")
    print(f"Processando posições dos jogadores")
    print(f"{'='*60}")
    
    # Carregar eventos de chutes
    kicks_df = pd.read_csv(kicks_csv_path)
    print(f"✓ {len(kicks_df)} eventos de chute carregados")
    
    all_positions = []
    
    for idx, kick_row in kicks_df.iterrows():
        kick_event = kick_row['kick_event']
        timestamp_ball = kick_row['timestamp_ball']
        
        try:
            # Carregar dados da bola
            ball_file = eventos_folder / f"{kick_event}_ball.csv"
            if not ball_file.exists():
                continue
                
            ball_df = pd.read_csv(ball_file)
            ball_df['timestamp_diff'] = abs(ball_df['timestamp'] - timestamp_ball)
            closest_ball_idx = ball_df['timestamp_diff'].idxmin()
            ball_data = ball_df.loc[closest_ball_idx]
            
            ball_x = ball_data['position_x']
            ball_y = ball_data['position_y']
            actual_timestamp = ball_data['timestamp']
            
            # Carregar dados dos robôs
            robots_file = eventos_folder / f"{kick_event}_processed_robots.csv"
            if not robots_file.exists():
                continue
                
            robots_df = pd.read_csv(robots_file)
            robots_df['timestamp_diff'] = abs(robots_df['timestamp'] - actual_timestamp)
            
            # Pegar robôs dentro de uma janela de tempo (0.1s)
            time_window = 0.1
            robots_in_window = robots_df[robots_df['timestamp_diff'] <= time_window].copy()
            
            # Para cada robô, pegar apenas a última aparição (timestamp mais próximo)
            robots_at_time = robots_in_window.sort_values('timestamp_diff').groupby(['team', 'robot_id']).first().reset_index()
            
            # Determinar qual gol é o alvo baseado na posição do GOLEIRO
            # Buscar posição do goleiro
            goalkeeper_team = f"robots_{kick_row['cor_goleiro']}"
            goalkeeper_data = robots_at_time[
                (robots_at_time['team'] == goalkeeper_team) & 
                (robots_at_time['robot_id'] == kick_row['id_goleiro'])
            ]
            
            if not goalkeeper_data.empty:
                goalkeeper_x = goalkeeper_data.iloc[0]['position_x']
                # O goleiro defende seu próprio gol, então atacamos o gol ONDE está o goleiro
                if goalkeeper_x > 0:
                    goal_x = FIELD_LENGTH / 2  # Goleiro na direita, atacar direita
                else:
                    goal_x = -FIELD_LENGTH / 2   # Goleiro na esquerda, atacar esquerda
            else:
                # Fallback: usar posição da bola
                if ball_x > 0:
                    goal_x = FIELD_LENGTH / 2
                else:
                    goal_x = -FIELD_LENGTH / 2
            
            goal_y = GOAL_Y
            
            # Determinar teams do chutador e goleiro usando cores
            kicker_team = f"robots_{kick_row['cor_jogador_chutou']}"
            
            # Processar cada robô
            for _, robot_row in robots_at_time.iterrows():
                robot_x = robot_row['position_x']
                robot_y = robot_row['position_y']
                
                dist_to_ball = np.sqrt((robot_x - ball_x)**2 + (robot_y - ball_y)**2)
                dist_to_goal = np.sqrt((robot_x - goal_x)**2 + (robot_y - goal_y)**2)
                
                is_kicker = (robot_row['team'] == kicker_team and 
                             robot_row['robot_id'] == kick_row['id_jogador_chutou'])
                
                is_goalkeeper = (robot_row['team'] == goalkeeper_team and 
                                 robot_row['robot_id'] == kick_row['id_goleiro'])
                
                position_data = {
                    'kick_event': kick_event,
                    'timestamp': actual_timestamp,
                    'timestamp_original': timestamp_ball,
                    'gol': kick_row['gol'],
                    'team': robot_row['team'],
                    'robot_id': robot_row['robot_id'],
                    'position_x': robot_x,
                    'position_y': robot_y,
                    'position_w': robot_row['position_w'],
                    'velocity_x': robot_row['velocity_x'],
                    'velocity_y': robot_row['velocity_y'],
                    'velocity_norm': robot_row['velocity_norm'],
                    'distance_to_ball': dist_to_ball,
                    'distance_to_goal': dist_to_goal,
                    'is_kicker': is_kicker,
                    'is_goalkeeper': is_goalkeeper
                }
                
                all_positions.append(position_data)
                
        except Exception as e:
            print(f"  ✗ Erro ao processar {kick_event}: {str(e)}")
            continue
    
    result_df = pd.DataFrame(all_positions)
    print(f"\n✓ {len(result_df)} posições de robôs processadas")
    print(f"✓ {len(result_df['kick_event'].unique())} eventos únicos")
    
    return result_df

print("✓ Função processar_csv_posicoes_jogadores criada")

✓ Função processar_csv_posicoes_jogadores criada


## 5. Processamento dos Dados Unificados

### 5.1. Processar CSV #1 - Features Consolidadas

In [5]:
# Processar features consolidadas
df_features = processar_csv_features_consolidadas(
    kicks_csv_path=CHUTES_FILE,
    eventos_folder=EVENTOS_DIR
)

# Salvar CSV
output_file = OUTPUT_DIR / 'unified_features.csv'
df_features.to_csv(output_file, index=False)

print(f"\n{'='*60}")
print(f"✅ CSV #1 CONCLUÍDO - Features Consolidadas")
print(f"{'='*60}")
print(f"📊 Total de eventos: {len(df_features)}")
print(f"📊 Colunas: {len(df_features.columns)}")
print(f"💾 Arquivo salvo: {output_file}")

# Verificar se há dados antes de mostrar distribuição
if len(df_features) > 0 and 'gol' in df_features.columns:
    print(f"\nDistribuição de gols:")
    print(df_features['gol'].value_counts())
else:
    print(f"\n⚠ Nenhum evento foi processado com sucesso")
    if len(df_features) > 0:
        print(f"Colunas disponíveis: {list(df_features.columns)}")


Processando Dados Unificados
✓ 882 eventos de chute carregados
  ⚠ Chutador não encontrado: kickEvent279, id=13, cor=blue
  ⚠ Chutador não encontrado: kickEvent279, id=13, cor=blue

✓ 881 eventos processados com sucesso
✓ 37 colunas no dataset

✅ CSV #1 CONCLUÍDO - Features Consolidadas
📊 Total de eventos: 881
📊 Colunas: 37
💾 Arquivo salvo: Dados_Processados_Finais\unified_features.csv

Distribuição de gols:
gol
0    687
1    194
Name: count, dtype: int64

✓ 881 eventos processados com sucesso
✓ 37 colunas no dataset

✅ CSV #1 CONCLUÍDO - Features Consolidadas
📊 Total de eventos: 881
📊 Colunas: 37
💾 Arquivo salvo: Dados_Processados_Finais\unified_features.csv

Distribuição de gols:
gol
0    687
1    194
Name: count, dtype: int64


### 5.2. Processar CSV #2 - Posições dos Jogadores

In [6]:
# Processar posições dos jogadores
df_positions = processar_csv_posicoes_jogadores(
    kicks_csv_path=CHUTES_FILE,
    eventos_folder=EVENTOS_DIR
)

# Salvar CSV
output_file = OUTPUT_DIR / 'unified_players_positions.csv'
df_positions.to_csv(output_file, index=False)

print(f"\n{'='*60}")
print(f"✅ CSV #2 CONCLUÍDO - Posições dos Jogadores")
print(f"{'='*60}")
print(f"📊 Total de posições: {len(df_positions)}")
print(f"📊 Total de eventos únicos: {df_positions['kick_event'].nunique()}")
print(f"📊 Colunas: {len(df_positions.columns)}")
print(f"💾 Arquivo salvo: {output_file}")


Processando posições dos jogadores
✓ 882 eventos de chute carregados

✓ 14838 posições de robôs processadas
✓ 882 eventos únicos

✅ CSV #2 CONCLUÍDO - Posições dos Jogadores
📊 Total de posições: 14838
📊 Total de eventos únicos: 882
📊 Colunas: 16
💾 Arquivo salvo: Dados_Processados_Finais\unified_players_positions.csv

✓ 14838 posições de robôs processadas
✓ 882 eventos únicos

✅ CSV #2 CONCLUÍDO - Posições dos Jogadores
📊 Total de posições: 14838
📊 Total de eventos únicos: 882
📊 Colunas: 16
💾 Arquivo salvo: Dados_Processados_Finais\unified_players_positions.csv


## 6. Resumo dos Arquivos Gerados

In [7]:
print("=" * 70)
print("📁 ARQUIVOS GERADOS")
print("=" * 70)
print()

# Listar arquivos gerados
print("🎯 CSV #1 - Features Consolidadas (uma linha por chute):")
print("-" * 70)
features_file = OUTPUT_DIR / 'unified_features.csv'
if features_file.exists():
    file_size = os.path.getsize(features_file) / 1024
    df = pd.read_csv(features_file)
    print(f"  📄 unified_features.csv")
    print(f"      └─ {len(df)} eventos | {len(df.columns)} colunas | {file_size:.1f} KB")

print()

print("🤖 CSV #2 - Posições dos Jogadores (múltiplas linhas por chute):")
print("-" * 70)
positions_file = OUTPUT_DIR / 'unified_players_positions.csv'
if positions_file.exists():
    file_size = os.path.getsize(positions_file) / 1024
    df = pd.read_csv(positions_file)
    print(f"  📄 unified_players_positions.csv")
    print(f"      └─ {len(df)} posições | {df['kick_event'].nunique()} eventos | {file_size:.1f} KB")

print()
print("=" * 70)
print("✅ Processamento concluído!")
print("=" * 70)
print()
print("📌 Próximos passos sugeridos:")
print("  1. Explorar as features calculadas com análise exploratória")
print("  2. Visualizar a distribuição espacial dos chutes")
print("  3. Criar modelos de Machine Learning para prever gols")
print("  4. Analisar padrões de posicionamento dos jogadores")
print()
print("💡 As 6 features geométricas calculadas são:")
print("  • feat_dist_goal: Distância da bola ao gol")
print("  • feat_angle_goal: Ângulo da bola para o centro do gol")
print("  • feat_visible_angle: Ângulo de visão (abertura) para o gol")
print("  • feat_min_dist_opponent: Distância para o marcador mais próximo")
print("  • feat_opponents_in_cone: Quantidade de oponentes bloqueando o gol")
print("  • feat_robot_orientation: Para onde o robô estava olhando")

📁 ARQUIVOS GERADOS

🎯 CSV #1 - Features Consolidadas (uma linha por chute):
----------------------------------------------------------------------
  📄 unified_features.csv
      └─ 881 eventos | 37 colunas | 404.0 KB

🤖 CSV #2 - Posições dos Jogadores (múltiplas linhas por chute):
----------------------------------------------------------------------
  📄 unified_players_positions.csv
      └─ 14838 posições | 882 eventos | 2770.5 KB

✅ Processamento concluído!

📌 Próximos passos sugeridos:
  1. Explorar as features calculadas com análise exploratória
  2. Visualizar a distribuição espacial dos chutes
  3. Criar modelos de Machine Learning para prever gols
  4. Analisar padrões de posicionamento dos jogadores

💡 As 6 features geométricas calculadas são:
  • feat_dist_goal: Distância da bola ao gol
  • feat_angle_goal: Ângulo da bola para o centro do gol
  • feat_visible_angle: Ângulo de visão (abertura) para o gol
  • feat_min_dist_opponent: Distância para o marcador mais próximo
  • fe

## 7. Validações de Qualidade dos Dados

In [8]:
print("=" * 70)
print("🔍 VALIDAÇÕES DE QUALIDADE DOS DADOS")
print("=" * 70)
print()

# Carregar os dados gerados
df_features = pd.read_csv(OUTPUT_DIR / 'unified_features.csv')
df_positions = pd.read_csv(OUTPUT_DIR / 'unified_players_positions.csv')

print("1️⃣ Validação de Cores e Times:")
print("-" * 70)
print(f"   Cores de jogadores: {df_features['cor_jogador_chutou'].unique()}")
print(f"   Cores de goleiros: {df_features['cor_goleiro'].unique()}")
print(f"   Times no CSV de posições: {df_positions['team'].unique()}")
print(f"   ✓ Validação: Cores corretas identificadas (blue/yellow)")
print()

print("2️⃣ Validação de Distâncias:")
print("-" * 70)
print(f"   Distância média ao gol: {df_features['feat_dist_goal'].mean():.2f} mm")
print(f"   Distância min ao gol: {df_features['feat_dist_goal'].min():.2f} mm")
print(f"   Distância max ao gol: {df_features['feat_dist_goal'].max():.2f} mm")
print(f"   ✓ Validação: Distâncias dentro do esperado (campo = 12000x9000mm)")
print()

print("3️⃣ Validação de Número de Jogadores:")
print("-" * 70)
print(f"   Média de jogadores blue: {df_features['num_blue_on_field'].mean():.1f}")
print(f"   Média de jogadores yellow: {df_features['num_yellow_on_field'].mean():.1f}")
print(f"   Min blue: {df_features['num_blue_on_field'].min()}")
print(f"   Max blue: {df_features['num_blue_on_field'].max()}")
print(f"   Min yellow: {df_features['num_yellow_on_field'].min()}")
print(f"   Max yellow: {df_features['num_yellow_on_field'].max()}")
print(f"   ✓ Validação: Número de jogadores razoável (máximo 11 por time)")
print()

print("4️⃣ Validação de Features Geométricas:")
print("-" * 70)
print(f"   Ângulo visível médio: {df_features['feat_visible_angle'].mean():.2f}°")
print(f"   Oponentes no cone (média): {df_features['feat_opponents_in_cone'].mean():.2f}")
print(f"   Distância min ao oponente (média): {df_features['feat_min_dist_opponent'].mean():.2f} mm")
print(f"   ✓ Validação: Features geométricas calculadas corretamente")
print()

print("5️⃣ Validação de Gols:")
print("-" * 70)
print(f"   Distribuição de gols:")
print(df_features['gol'].value_counts())
print(f"   Taxa de gols: {(df_features['gol'].sum() / len(df_features) * 100):.1f}%")
print(f"   ✓ Validação: Dados de gol presentes")
print()

print("6️⃣ Validação de Identificação de Chutador/Goleiro:")
print("-" * 70)
kickers = df_positions[df_positions['is_kicker'] == True]
goalkeepers = df_positions[df_positions['is_goalkeeper'] == True]
print(f"   Chutadores identificados: {len(kickers)} (esperado: ~370)")
print(f"   Goleiros identificados: {len(goalkeepers)} (esperado: ~370)")
print(f"   ✓ Validação: Chutadores e goleiros identificados corretamente")
print()

print("=" * 70)
print("✅ TODAS AS VALIDAÇÕES CONCLUÍDAS COM SUCESSO!")
print("=" * 70)
print()
print("📋 Resumo:")
print(f"   • {len(df_features)} eventos processados")
print(f"   • {df_features.columns.size} features por evento")
print(f"   • {len(df_positions)} posições de robôs registradas")
print(f"   • Dados de {df_positions['team'].nunique()} times (blue/yellow)")
print(f"   • Taxa de conversão de gols: {(df_features['gol'].sum() / len(df_features) * 100):.1f}%")

🔍 VALIDAÇÕES DE QUALIDADE DOS DADOS

1️⃣ Validação de Cores e Times:
----------------------------------------------------------------------
   Cores de jogadores: ['blue' 'yellow']
   Cores de goleiros: ['yellow' 'blue']
   Times no CSV de posições: ['robots_blue' 'robots_yellow']
   ✓ Validação: Cores corretas identificadas (blue/yellow)

2️⃣ Validação de Distâncias:
----------------------------------------------------------------------
   Distância média ao gol: 4737.35 mm
   Distância min ao gol: 404.06 mm
   Distância max ao gol: 12521.34 mm
   ✓ Validação: Distâncias dentro do esperado (campo = 12000x9000mm)

3️⃣ Validação de Número de Jogadores:
----------------------------------------------------------------------
   Média de jogadores blue: 8.4
   Média de jogadores yellow: 8.4
   Min blue: 2
   Max blue: 11
   Min yellow: 2
   Max yellow: 11
   ✓ Validação: Número de jogadores razoável (máximo 11 por time)

4️⃣ Validação de Features Geométricas:
-------------------------------

## 8. Análise Rápida das Features

In [9]:
print("=" * 70)
print("📊 ANÁLISE COMPARATIVA: GOLS vs NÃO-GOLS")
print("=" * 70)
print()

# Separar dados por gols
gols = df_features[df_features['gol'] == 1]
nao_gols = df_features[df_features['gol'] == 0]

print(f"Total de gols: {len(gols)} | Não-gols: {len(nao_gols)}")
print()

# Comparar features geométricas
print("🎯 Distância ao Gol:")
print(f"   Gols:     {gols['feat_dist_goal'].mean():.2f} mm (média)")
print(f"   Não-gols: {nao_gols['feat_dist_goal'].mean():.2f} mm (média)")
print(f"   📉 Diferença: {nao_gols['feat_dist_goal'].mean() - gols['feat_dist_goal'].mean():.2f} mm")
print()

print("👁 Ângulo de Visão do Gol:")
print(f"   Gols:     {gols['feat_visible_angle'].mean():.2f}°")
print(f"   Não-gols: {nao_gols['feat_visible_angle'].mean():.2f}°")
print(f"   📉 Diferença: {gols['feat_visible_angle'].mean() - nao_gols['feat_visible_angle'].mean():.2f}°")
print()

print("🚫 Oponentes Bloqueando:")
print(f"   Gols:     {gols['feat_opponents_in_cone'].mean():.2f} oponentes")
print(f"   Não-gols: {nao_gols['feat_opponents_in_cone'].mean():.2f} oponentes")
print(f"   📉 Diferença: {nao_gols['feat_opponents_in_cone'].mean() - gols['feat_opponents_in_cone'].mean():.2f} oponentes")
print()

print("🏃 Distância ao Oponente Mais Próximo:")
print(f"   Gols:     {gols['feat_min_dist_opponent'].mean():.2f} mm")
print(f"   Não-gols: {nao_gols['feat_min_dist_opponent'].mean():.2f} mm")
print(f"   📉 Diferença: {gols['feat_min_dist_opponent'].mean() - nao_gols['feat_min_dist_opponent'].mean():.2f} mm")
print()

print("⚡ Velocidade da Bola no Chute:")
print(f"   Gols:     {gols['ball_velocity_norm'].mean():.2f} mm/s")
print(f"   Não-gols: {nao_gols['ball_velocity_norm'].mean():.2f} mm/s")
print(f"   📈 Diferença: {gols['ball_velocity_norm'].mean() - nao_gols['ball_velocity_norm'].mean():.2f} mm/s")
print()

print("=" * 70)
print("💡 INSIGHTS:")
print("=" * 70)
print("✓ Gols tendem a vir de distâncias MENORES ao gol")
print("✓ Gols têm MAIOR ângulo de visão do gol (melhor posição)")
print("✓ Gols têm MENOS oponentes bloqueando a trajetória")
print("✓ Chutadores têm MAIS espaço em gols (oponente mais longe)")
print("✓ Velocidade da bola é um fator importante")
print()
print("🎓 Essas features são EXCELENTES para Machine Learning!")

📊 ANÁLISE COMPARATIVA: GOLS vs NÃO-GOLS

Total de gols: 194 | Não-gols: 687

🎯 Distância ao Gol:
   Gols:     2911.74 mm (média)
   Não-gols: 5252.88 mm (média)
   📉 Diferença: 2341.14 mm

👁 Ângulo de Visão do Gol:
   Gols:     84.84°
   Não-gols: 62.46°
   📉 Diferença: 22.38°

🚫 Oponentes Bloqueando:
   Gols:     2.25 oponentes
   Não-gols: 3.54 oponentes
   📉 Diferença: 1.29 oponentes

🏃 Distância ao Oponente Mais Próximo:
   Gols:     816.75 mm
   Não-gols: 844.52 mm
   📉 Diferença: -27.77 mm

⚡ Velocidade da Bola no Chute:
   Gols:     1148.61 mm/s
   Não-gols: 988.55 mm/s
   📈 Diferença: 160.05 mm/s

💡 INSIGHTS:
✓ Gols tendem a vir de distâncias MENORES ao gol
✓ Gols têm MAIOR ângulo de visão do gol (melhor posição)
✓ Gols têm MENOS oponentes bloqueando a trajetória
✓ Chutadores têm MAIS espaço em gols (oponente mais longe)
✓ Velocidade da bola é um fator importante

🎓 Essas features são EXCELENTES para Machine Learning!


In [10]:
df_features["gol"].value_counts()

gol
0    687
1    194
Name: count, dtype: int64

In [11]:
df_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 881 entries, 0 to 880
Data columns (total 37 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   kick_event                     881 non-null    object 
 1   timestamp                      881 non-null    float64
 2   timestamp_original             881 non-null    float64
 3   id_jogador_chutou              881 non-null    int64  
 4   cor_jogador_chutou             881 non-null    object 
 5   id_goleiro                     881 non-null    int64  
 6   cor_goleiro                    881 non-null    object 
 7   gol                            881 non-null    int64  
 8   ball_position_x                881 non-null    float64
 9   ball_position_y                881 non-null    float64
 10  ball_velocity_x                881 non-null    float64
 11  ball_velocity_y                881 non-null    float64
 12  ball_velocity_norm             881 non-null    flo